In [7]:
from openai import OpenAI

client = OpenAI()

MODEL = "gpt-6-luna"

agent = client.beta.agents.create(
    model=MODEL,
    instructions="You speak short sentences. Like caveman.",
    reasoning={"effort": "low"},
    text={"verbosity": "low"},    
)

In [8]:
import base64

data = b"name:Ax School\nCourses:1000\n"

with client.beta.agents.sessions.create(
    agent_id=agent.id,
    environment={
        "type": "openai_hosted",
        "container_size": "small",
        "packages": {"python": ["PyYAML==6.0.3"]},
        "network": {
            "access": "disabled",
        },
        "setup_commands": [{"command": "mkdir -p outputs"}],
        "files": [
            {
                "type": "inline",
                "path": "/workspace/course.yaml",
                "data": base64.b64encode(data).decode(),
            }
        ],
    },
    input="Tell me the files i have in my /workspace if they are .yml open them, and print the parsed object. And save it as JSON to /workspace/outputs/course.json then show me the output of that file.",
    stream=True,
) as events:
    for event in events:
        if event.type == "agent.session.created":
            session_id = event.session.id
        elif event.type == "agent.session.requires_action":
            print(event.session.required_actions)
            break
        elif event.type == "agent.session.turn.output_text.done":
            print(event.text)
        elif (
            event.type == "agent.session.turn.item.done"
            and event.item.type == "command_execution"
        ):
            print("Command: ", event.item.command)
            print("Output: ", event.item.output)
        elif event.type == "agent.session.turn.output_text.delta":
            continue
        else:
            print(event.type)

agent.session.turn.created
agent.session.turn.item.added
agent.session.in_progress
agent.session.turn.in_progress
agent.session.turn.item.added
agent.session.turn.content_part.added
I’ll list `/workspace`, read each `.yml` file, parse it, then save and show the JSON.
agent.session.turn.content_part.done
agent.session.turn.item.done
agent.session.environment.ready
agent.session.environment.connected
agent.session.turn.item.added
Command:  /bin/bash -lc "find /workspace -maxdepth 3 -type f -printf '%p\\n' | sort"
Output:  /workspace/.managed-agents-executor.log
/workspace/.managed-agents-executor.pid
/workspace/.managed-agents/setup/started_at
/workspace/.managed-agents/setup/state
/workspace/course.yaml

agent.session.turn.item.added
agent.session.turn.item.done
agent.session.turn.item.added
Command:  /bin/bash -lc "cat /workspace/course.yaml && python - <<'PY'
import yaml
from pathlib import Path
p=Path('/workspace/course.yaml')
obj=yaml.safe_load(p.read_text())
print('PARSED:', repr(o

In [9]:
artifacts = client.beta.agents.sessions.artifacts.list(session_id)

for artifact in artifacts:
    artifact_id = artifact.id
    print(artifact.to_dict())
    content = client.beta.agents.sessions.artifacts.content(
        artifact_id, session_id=session_id
    )
    print(content.content)

{'id': 'artifact_f30f2ca0a7034e67adc2f50d9c92c8d7e24a35d97d354355aa', 'created_at': 1793284482, 'environment_id': 'ccarenv_b64_Y2NhcmVudl82YWUzNTk1NGU5ZDg4MTkxYWYwYTY0MDk4YjI2MTlkZA', 'object': 'agent.session.artifact', 'path': '/workspace/outputs/course.json', 'session_id': 'sess_021ab63bc15ef82f006ae3595483e081989929ab48806e7cfa', 'size_bytes': 30, 'turn_id': 'turn_021ab63bc15ef82f006ae35959c6408198b45fb2c4ebdfe685'}
b'"name:Ax School Courses:1000"\n'


In [10]:
session = client.beta.agents.sessions.retrieve(session_id)

environment = client.beta.agents.environments.retrieve(session.environment.id)

upload = client.beta.agents.environments.files.create(
    environment_id=environment.id,
    type="inline",
    path="/workspace/hello.txt",
    data=base64.b64encode(b"Hello world lol\n").decode(),
)

upload

EnvironmentFile(environment_id='ccarenv_b64_Y2NhcmVudl82YWUzNTk1NGU5ZDg4MTkxYWYwYTY0MDk4YjI2MTlkZA', object='agent.environment.file', path='/workspace/hello.txt', size_bytes=16)

In [11]:
with client.beta.agents.sessions.events.stream(session_id) as events:
    client.beta.agents.sessions.events.create(
        session_id,
        events=[
            {
                "type": "agent.session.input.message",
                "input": [
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "input_text",
                                "text": "Tell me the files in my workspace again",
                            }
                        ],
                    }
                ],
            }
        ],
    )
    for event in events:        
        if event.type == "agent.session.turn.output_text.delta":
            continue
        elif event.type == "agent.session.created":
            session_id = event.session.id
        elif event.type == "agent.session.turn.output_text.done":
            print(event.text)
        elif (
            event.type == "agent.session.turn.item.done"
            and event.item.type == "command_execution"
        ):
            print("Command: ", event.item.command)
            print("Output: ", event.item.output)
        elif event.type == "agent.session.turn.completed":
            print(event.type)
            break
        else:
            print(event.type)
            


agent.session.environment.ready
agent.session.turn.created
agent.session.turn.item.added
agent.session.in_progress
agent.session.turn.in_progress
agent.session.turn.item.added
agent.session.turn.content_part.added
I’ll check the current workspace file list.
agent.session.turn.content_part.done
agent.session.turn.item.done
agent.session.turn.item.added
Command:  /bin/bash -lc "find /workspace -type f -printf '%p\\n' | sort"
Output:  /workspace/.managed-agents-executor.log
/workspace/.managed-agents-executor.pid
/workspace/.managed-agents/setup/started_at
/workspace/.managed-agents/setup/state
/workspace/course.yaml
/workspace/hello.txt
/workspace/outputs/course.json

agent.session.turn.item.added
agent.session.turn.content_part.added
Files in `/workspace`:

- `.managed-agents-executor.log`
- `.managed-agents-executor.pid`
- `.managed-agents/setup/started_at`
- `.managed-agents/setup/state`
- `course.yaml`
- `hello.txt`
- `outputs/course.json`
agent.session.turn.content_part.done
agent.s